# Tabular Data Playground — JupyterLite Demo

This notebook runs entirely in your browser using [JupyterLite](https://jupyterlite.readthedocs.io/) and the [Pyodide](https://pyodide.org/) kernel — no server, no account required.

It loads a small sample dataset of European capital cities and produces several matplotlib visualisations inline.

**Run All** (`Kernel → Restart Kernel and Run All Cells`) to execute everything in one go.

## 1. Install and import dependencies

The cell below installs `matplotlib` from the Pyodide wheel index. This happens in-browser — no local Python installation is needed.

In [ ]:
%pip install matplotlib

In [ ]:
import csv
import io
import matplotlib
import matplotlib.pyplot as plt

print(f"matplotlib {matplotlib.__version__} loaded successfully")

## 2. Load the sample dataset

The CSV file ships with this JupyterLite site and is read from the virtual filesystem — no network request needed.

In [ ]:
rows = []
with open("data/sample.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

# Convert numeric columns
for row in rows:
    row["population"] = int(row["population"])
    row["gdp_per_capita_usd"] = float(row["gdp_per_capita_usd"])
    row["life_expectancy"] = float(row["life_expectancy"])

print(f"Loaded {len(rows)} rows")
print("Columns:", list(rows[0].keys()))
print("\nFirst 3 rows:")
for r in rows[:3]:
    print(" ", r)

## 3. Bar chart — GDP per capita by city

In [ ]:
# Sort by GDP descending, show top 15
top = sorted(rows, key=lambda r: r["gdp_per_capita_usd"], reverse=True)[:15]
cities = [r["city"] for r in top]
gdp = [r["gdp_per_capita_usd"] / 1000 for r in top]  # in thousands

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cities[::-1], gdp[::-1], color="steelblue")
ax.set_xlabel("GDP per capita (USD thousands)")
ax.set_title("Top 15 European capitals by GDP per capita (2022)")
ax.bar_label(bars, fmt="%.0fk", padding=4)
fig.tight_layout()
plt.show()

## 4. Scatter plot — GDP vs Life Expectancy

In [ ]:
gdp_vals = [r["gdp_per_capita_usd"] / 1000 for r in rows]
le_vals = [r["life_expectancy"] for r in rows]
labels = [r["city"] for r in rows]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(gdp_vals, le_vals, color="coral", edgecolors="k", linewidths=0.5, s=60, zorder=3)

# Label a few notable cities
highlight = {"Luxembourg City", "Oslo", "Reykjavik", "Athens", "Bucharest", "Sofia"}
for r in rows:
    if r["city"] in highlight:
        ax.annotate(
            r["city"],
            (r["gdp_per_capita_usd"] / 1000, r["life_expectancy"]),
            textcoords="offset points", xytext=(5, 3), fontsize=8
        )

ax.set_xlabel("GDP per capita (USD thousands)")
ax.set_ylabel("Life expectancy (years)")
ax.set_title("GDP per capita vs Life Expectancy — European capitals (2022)")
ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()

## 5. Population by country

In [ ]:
# Group by country
by_country = {}
for r in rows:
    c = r["country"]
    by_country[c] = by_country.get(c, 0) + r["population"]

# Sort by population and show top 10
countries = sorted(by_country, key=by_country.get, reverse=True)[:10]
pops = [by_country[c] / 1e6 for c in countries]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(countries, pops, color="mediumseagreen", edgecolor="k", linewidth=0.5)
ax.set_ylabel("Capital city population (millions)")
ax.set_title("Top 10 countries by capital city population (sample dataset, 2022)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

---

All three charts were rendered **entirely in your browser** using Pyodide + matplotlib, with no server involved. The source data is the small CSV shipped alongside this notebook.

Return to the [Frictionless Data Explorer](../) to explore the full IDE and lessons.